# Notebook 04 — Optional EDGAR Data Acquisition Appendix

This notebook documents the data acquisition process used to build the EDGAR-based financial statement dataset for the Corporate Credit Clustering Tool.

The main credit-risk modelling and scoring workflow is completed in the previous notebooks. This notebook should therefore be read as a technical appendix and reusable data-engineering tool. Its purpose is to explain how the raw public-company universe is created, how EDGAR data is collected, how industry information is added, and how the raw facts are converted into a queryable analytical layer for model training.

The workflow is intentionally split into safe, explicit steps because SEC EDGAR data acquisition can become heavy very quickly. A small test run and a full refresh are very different operations. For that reason, the notebook uses execution flags, resumable checkpoints, CSV/parquet storage, and DuckDB views instead of trying to run everything automatically in one pass.

The notebook has four practical goals:

1. build and refresh a ticker / CIK / SIC company universe;
2. enrich the universe with SEC industry classification information;
3. download raw EDGAR financial facts in a controlled and resumable way;
4. convert the raw outputs into parquet and DuckDB structures that can be used by the modelling notebooks.

This notebook also preserves the data-acquisition logic for future development. The model can only be refreshed or expanded if the underlying EDGAR pipeline remains reproducible.

## 1. Imports

This cell loads the libraries needed for the EDGAR data-acquisition workflow.

The notebook uses `edgartools` to access SEC company information and financial facts, `pandas` for tabular processing, `requests` and `read_html()` for the SIC table, `tqdm` for progress tracking, and `DuckDB` for querying large parquet-backed datasets without loading everything into memory.

`nest_asyncio` is applied because some notebook environments already run an event loop, and EDGAR-related tools may need to work safely inside that environment. The goal of this first step is only to prepare the notebook environment; no EDGAR data is downloaded yet.

In [ ]:
from datetime import date
from pathlib import Path

import nest_asyncio

# from pytickersymbols import PyTickerSymbols
from edgar import *

nest_asyncio.apply()

import glob
import os
from io import StringIO

import duckdb
import pandas as pd
import requests
from IPython.display import display
from tqdm import tqdm

## 2. Notebook configuration and project paths

This cell defines the project folder structure and the main file paths used throughout the notebook.

The notebook is expected to run from the `notebooks/` directory, so the repository root is set as one folder above the current working directory. All generated data is stored under the root-level `data/` folder. This keeps the repository organized and avoids scattering raw EDGAR outputs across the project.

The path structure separates raw, interim, processed, parquet, and DuckDB files:

- `data/raw/` stores the first downloaded or scraped outputs;
- `data/interim/` stores intermediate checkpoints;
- `data/processed/` stores cleaned and enriched datasets used later;
- `data/raw_financial_facts_parquet/` stores parquet chunks created from the large raw facts file;
- `data/duckdb/` stores the DuckDB analytical database.

This organization matters because EDGAR outputs can become large. The notebook needs predictable locations so that later cells can resume work, reuse checkpoints, and avoid repeating expensive downloads.

In [ ]:
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 120)

# This notebook is expected to be run from the notebooks/ directory.
PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
PARQUET_DIR = DATA_DIR / "raw_financial_facts_parquet"
DB_DIR = DATA_DIR / "duckdb"

for folder in [DATA_DIR, RAW_DIR, INTERIM_DIR, PROCESSED_DIR, PARQUET_DIR, DB_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

FUNDAMENTAL_UNIVERSE_PATH = RAW_DIR / "fundamental_universe.csv"
TICKER_CIK_SIC_PATH = INTERIM_DIR / "fundamental_universe_ticker_cik_sic.csv"
TICKER_SIC_INDUSTRY_PATH = (
    PROCESSED_DIR / "03_fundamental_universe_ticker_sic_industry.csv"
)
RAW_FACTS_CSV_PATH = RAW_DIR / "raw_financial_facts.csv"
DUCKDB_PATH = DB_DIR / "financials.duckdb"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 3. Execution flags

This cell controls which parts of the notebook are allowed to run.

This is one of the most important safety mechanisms in the EDGAR pipeline. Some cells only perform small sanity checks, while others can call EDGAR repeatedly or process very large local files. For that reason, the notebook does not assume that every cell should run every time.

The flags make the workflow explicit:

- `RUN_EDGAR_SANITY_CHECK` tests whether EDGAR access works;
- `RUN_TICKER_UNIVERSE_BUILD` builds the ticker / CIK / SIC universe;
- `RUN_SIC_TABLE_DOWNLOAD` downloads the SEC SIC table;
- `RUN_SIC_ENRICHMENT` merges SIC labels into the company universe;
- `RUN_RAW_EDGAR_FACT_DOWNLOAD` performs the heavy raw financial fact download;
- `RUN_CSV_TO_PARQUET` converts the raw CSV into parquet chunks;
- `RUN_INCREMENTAL_REFRESH_AUDIT` checks whether the local dataset may need a refresh.

`MAX_TICKERS_FOR_DOWNLOAD` is used for small test runs. A full refresh should only be run intentionally. This is why the notebook also requires `CONFIRM_FULL_EDGAR_DOWNLOAD = True` before downloading all tickers. This prevents an accidental full EDGAR run during casual notebook review.

I am leaving the notebook configured for a test run for reproducibility showcase.

In [ ]:
RUN_EDGAR_SANITY_CHECK = True
RUN_TICKER_UNIVERSE_BUILD = True
RUN_SIC_TABLE_DOWNLOAD = True
RUN_SIC_ENRICHMENT = True
RUN_RAW_EDGAR_FACT_DOWNLOAD = True
RUN_CSV_TO_PARQUET = True
RUN_INCREMENTAL_REFRESH_AUDIT = False

# ---------------------------------------------------------------------
# Test / full-run control
# ---------------------------------------------------------------------

TEST_RUN_MODE = True

# Controls the slow ticker-universe enrichment step.
# Set to None only for full universe build.
MAX_TICKERS_FOR_UNIVERSE_BUILD = 25

# Controls the raw EDGAR facts download step.
# Set to None only for full raw-facts download.
MAX_TICKERS_FOR_DOWNLOAD = 10

MAX_TICKERS_FOR_DOWNLOAD = (
    10  # Use a small integer for testing; set to None only for full refresh.
)


CONFIRM_FULL_EDGAR_DOWNLOAD = False

TARGET_MIN_FISCAL_YEAR = 2020
TARGET_MAX_FISCAL_YEAR = 2026

if TEST_RUN_MODE:
    FUNDAMENTAL_UNIVERSE_PATH = RAW_DIR / "fundamental_universe_TEST.csv"
    TICKER_CIK_SIC_PATH = INTERIM_DIR / "fundamental_universe_ticker_cik_sic_TEST.csv"
    TICKER_SIC_INDUSTRY_PATH = (
        PROCESSED_DIR / "03_fundamental_universe_ticker_sic_industry_TEST.csv"
    )
    RAW_FACTS_CSV_PATH = RAW_DIR / "raw_financial_facts_TEST.csv"
    PARQUET_DIR = DATA_DIR / "raw_financial_facts_parquet_TEST"
    DUCKDB_PATH = DB_DIR / "financials_TEST.duckdb"

    PARQUET_DIR.mkdir(parents=True, exist_ok=True)

    print("TEST_RUN_MODE is active. Test files will be used.")
else:
    print("FULL_RUN_MODE is active. Production files will be used.")

## 4. EDGAR access sanity check

This cell checks whether the notebook can access EDGAR data and whether local EDGAR storage is working.

The check is intentionally lightweight. It calls `get_company_tickers()` and displays a small sample of the available ticker universe. If this works, the notebook has a functioning connection to the public SEC company ticker dataset and can proceed to later data-acquisition steps.

This is useful before running heavier cells. If the sanity check fails, there is no point in trying to build the full universe or download financial facts. It also helps separate connection or library issues from problems in the project logic itself.

In [ ]:
if RUN_EDGAR_SANITY_CHECK:
    use_local_storage(True)

    try:
        tickers = get_company_tickers()
        df_tickers = tickers.to_pandas() if hasattr(tickers, "to_pandas") else tickers

        print("Success. Local EDGAR storage connected.")
        print(f"Total tickers: {len(df_tickers)}")
        display(df_tickers.head())
        print(df_tickers.columns)

    except Exception as e:
        print(f"EDGAR sanity check failed: {e}")
else:
    print("EDGAR sanity check skipped. Set RUN_EDGAR_SANITY_CHECK=True to execute.")

## 5. Optional company-level probe

This cell is a small exploratory probe of the `Company` object from `edgartools`.

I use a few well-known tickers only to understand what information the object exposes and whether key methods return useful accounting data. The example checks tickers such as `A`, `WMT`, and `JPM`, then tries to retrieve company facts and total assets.

This is not part of the production data pipeline. It is a development and learning cell. It helped me understand the `edgartools` interface before building the larger raw-fact download workflow.

In [ ]:
if RUN_EDGAR_SANITY_CHECK:
    from edgar import Company

    for t in ["A", "WMT", "JPM"]:
        try:
            c = Company(t)
            facts = c.get_facts()

            assets = facts.get_total_assets()

            print(f"Ticker: {t:4} | SIC: {c.sic} | Assets: ${assets:,.0f}")

        except Exception as e:
            print(f"Error for {t}: {e}")
else:
    print("Company-level EDGAR probe skipped.")

## 6. Build the fundamental ticker universe

This step builds the initial company universe used for later EDGAR fact extraction.

The output is a ticker / CIK / SIC checkpoint saved to `data/raw/fundamental_universe.csv`. This is an EDGAR-calling step, so it is disabled by default and controlled by `RUN_TICKER_UNIVERSE_BUILD`.

The cell is designed to be resumable. If the output file already exists, the notebook reads the existing checkpoint and skips tickers that were already processed. This is important because EDGAR workflows can be interrupted by network issues, local runtime limits, or user decisions to stop early.

The purpose of this step is not yet to build financial ratios. It only creates the base company reference table: ticker, company identifier, company name, and SIC classification. Later steps enrich this universe and use it to download financial facts.

In [ ]:
if RUN_TICKER_UNIVERSE_BUILD:
    use_local_storage(True)

    df = get_company_tickers()
    df = df.to_pandas() if hasattr(df, "to_pandas") else df

    output_file = FUNDAMENTAL_UNIVERSE_PATH

    # Defensive ticker column normalization.
    if "ticker" not in df.columns and "Ticker" in df.columns:
        df = df.rename(columns={"Ticker": "ticker"})

    ticker_list = df["ticker"].dropna().astype(str).tolist()

    if TEST_RUN_MODE:
        ticker_list = ticker_list[:MAX_TICKERS_FOR_UNIVERSE_BUILD]
        print(
            f"TEST_RUN_MODE enabled. Building ticker universe for "
            f"{len(ticker_list)} tickers."
        )
    else:
        print(f"FULL RUN enabled. Building ticker universe for {len(ticker_list)} tickers.")

    # Resume if file already exists.
    if output_file.exists():
        final_df = pd.read_csv(output_file)
        completed = set(final_df["ticker"].astype(str).tolist())
        print(f"Resuming... already completed: {len(completed)}")
    else:
        final_df = pd.DataFrame()
        completed = set()

    rows = []

    for ticker in tqdm(ticker_list):
        if ticker in completed:
            continue

        try:
            c = Company(ticker)
            facts = c.get_facts()

            rows.append(
                {
                    "ticker": ticker,
                    "cik": c.cik,
                    "name": c.name,
                    "sic": c.sic,
                    "sic_description": getattr(c, "sic_description", None),
                    "total_assets": facts.get_total_assets(),
                    "total_liabilities": facts.get_total_liabilities(),
                    "revenue": facts.get_revenue(),
                    "net_income": facts.get_net_income(),
                }
            )

            # Save every 100 rows.
            if len(rows) >= 100:
                temp_df = pd.DataFrame(rows)
                final_df = pd.concat([final_df, temp_df], ignore_index=True)
                final_df.to_csv(output_file, index=False)
                rows = []

        except Exception as e:
            print(f"Skipped {ticker}: {e}")
            continue

    # Final save.
    if rows:
        temp_df = pd.DataFrame(rows)
        final_df = pd.concat([final_df, temp_df], ignore_index=True)
        final_df.to_csv(output_file, index=False)

    print("Done.")
    print(f"Total saved: {len(final_df)}")
    print(f"Output file: {output_file}")
else:
    print(
        "Ticker universe build skipped. Set RUN_TICKER_UNIVERSE_BUILD=True to execute."
    )

## 7. Download the SEC SIC table

This step downloads the SEC Standard Industrial Classification table.

SIC codes are important because the model needs to know which companies are financial and which are non-financial. The final clustering model intentionally focuses on non-financial companies, so industry classification is part of the methodological control of the project.

The SEC SIC table provides the human-readable industry title for each SIC code. This cell retrieves the table, parses it with `pandas.read_html()`, standardizes the SIC column name, and prepares it for merging with the ticker universe.

The `User-Agent` header should be replaced with a real contact email before running against SEC resources. This is part of responsible EDGAR access practice.

In [ ]:
if RUN_SIC_TABLE_DOWNLOAD:
    url = "https://www.sec.gov/corpfin/division-of-corporation-finance-standard-industrial-classification-sic-code-list"

    headers = {
        "User-Agent": "your_email@domain.com",  # Replace with your real contact email before running.
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Host": "www.sec.gov",
    }

    response = requests.get(url, headers=headers)
    print("SEC response status:", response.status_code)
    response.raise_for_status()

    html_io = StringIO(response.text)
    tables = pd.read_html(html_io)

    sic_df = tables[0].copy()

    print("Original SIC table columns:")
    print(sic_df.columns.tolist())
    display(sic_df.head())

    # SEC page is currently parsed with numeric columns [0, 1, 2],
    # and the real header appears as the first data row.
    if list(sic_df.columns) == [0, 1, 2]:
        sic_df.columns = sic_df.iloc[0]
        sic_df = sic_df.iloc[1:].reset_index(drop=True)

    # Normalize column names.
    sic_df.columns = (
        sic_df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("-", "_")
    )

    print("Normalized SIC table columns:")
    print(sic_df.columns.tolist())

    # Rename expected columns.
    sic_df = sic_df.rename(
        columns={
            "sic_code": "sic",
            "industry_title": "Industry Title",
        }
    )

    if "sic" not in sic_df.columns:
        raise KeyError(
            f"SIC column not found after normalization. Available columns: {sic_df.columns.tolist()}"
        )

    # Normalize SIC values.
    sic_df["sic"] = pd.to_numeric(sic_df["sic"], errors="coerce")
    sic_df = sic_df.dropna(subset=["sic"])
    sic_df["sic"] = sic_df["sic"].astype(int)

    sic_df.info()
    display(sic_df.head())

else:
    print("SIC table download skipped. Set RUN_SIC_TABLE_DOWNLOAD=True to execute.")

## 8. Enrich the universe with SIC industry titles

This cell merges the fundamental ticker universe with the SEC SIC industry table.

The ticker universe gives the company identifier and SIC code. The SIC table adds the industry title. The merged result becomes a more useful analytical checkpoint because each company is now connected not only to a numeric SIC code, but also to a readable industry label.

The output is saved to `data/interim/fundamental_universe_ticker_cik_sic.csv`. This intermediate checkpoint is useful because later cells can continue from it without repeating the EDGAR universe build or SIC table download.

In [ ]:
if RUN_SIC_ENRICHMENT:
    if not FUNDAMENTAL_UNIVERSE_PATH.exists():
        raise FileNotFoundError(
            f"Fundamental universe file not found: {FUNDAMENTAL_UNIVERSE_PATH}. "
            "Run ticker universe build first or disable SIC enrichment."
        )

    fundamental_df = pd.read_csv(FUNDAMENTAL_UNIVERSE_PATH)
    fundamental_df = fundamental_df.dropna(subset=["sic", "ticker"])

    fundamental_df = fundamental_df.merge(
        sic_df,
        how="left",
        on="sic",
    )

    if "sic_description" in fundamental_df.columns:
        fundamental_df = fundamental_df.drop(columns=["sic_description"])

    fundamental_df.to_csv(TICKER_CIK_SIC_PATH, index=False)

    display(fundamental_df.head())
else:
    print("SIC enrichment skipped. Set RUN_SIC_ENRICHMENT=True to execute.")

## 9. Load the enriched ticker universe checkpoint

From this point onward, the notebook reads the saved enriched universe checkpoint instead of calling EDGAR again.

This is a key design decision. Once the ticker / CIK / SIC universe has been created and enriched, later audit and preparation steps should operate from local files. This makes the notebook faster, more reproducible, and safer to review.

The displayed table confirms the structure of the enriched universe before the notebook performs SIC coverage checks and industry mapping.

In [ ]:
fundamental_df = pd.read_csv(TICKER_CIK_SIC_PATH)
fundamental_df

## 10. Audit SIC coverage

This section checks whether the enriched universe has usable SIC and industry coverage.

Before downloading raw financial facts, it is important to understand the company universe. If many companies are missing SIC codes or industry titles, the later segmentation into financial and non-financial companies would be weaker.

The following cells inspect SIC code frequency, industry-title frequency, a grouped SIC audit table, and a single ticker example. These checks are simple, but they are useful governance controls before the project commits to a large EDGAR download.

In [ ]:
fundamental_df["sic"].value_counts()

### Industry-title distribution

This cell checks the frequency of readable SIC industry titles. It complements the numeric SIC audit by showing the industry descriptions that will later help with interpretation and sector diagnostics.

In [ ]:
fundamental_df["Industry Title"].value_counts()

### SIC audit table

This cell groups the company universe by SIC code and industry title. It counts companies and gives sample tickers and names for each SIC category. The goal is to make the SIC coverage auditable before moving further into the EDGAR fact extraction workflow.

In [ ]:
sic_audit = (
    fundamental_df.groupby(["sic", "Industry Title"])
    .agg(
        company_count=("ticker", "count"),
        sample_tickers=("ticker", lambda x: ", ".join(x.astype(str).head(10))),
        sample_names=("name", lambda x: " | ".join(x.astype(str).head(5))),
    )
    .reset_index()
    .sort_values("company_count", ascending=False)
)

sic_audit

### Single-company SIC spot check

This small cell checks one known ticker in the enriched universe. It is a practical sanity check that the ticker lookup, SIC code, and related fields are working as expected.

In [ ]:
fundamental_df[fundamental_df["ticker"] == "ADP"]

## 11. Add broad SIC major-division buckets

The SEC SIC titles are detailed and can produce many industry categories. For the credit-clustering project, I also need broader sector-style groups that are easier to interpret and use in diagnostics.

This step defines a mapping from SIC numeric ranges to broad major divisions, such as Manufacturing / Industrials, Services, Transportation / Utilities, Wholesale / Retail, and Finance / Insurance / Real Estate.

This mapping is especially important because the final KMeans model is trained only on non-financial issuers. Financial, insurance, and real estate companies are identified and later excluded from the core non-financial clustering scope because they require specialized credit-risk features.

In [ ]:
sic_major_division_map = {
    range(100, 1000): "Agriculture",
    range(1000, 1500): "Mining / Energy",
    range(1500, 1800): "Construction",
    range(2000, 4000): "Manufacturing / Industrials",
    range(4000, 5000): "Transportation / Utilities",
    range(5000, 6000): "Wholesale / Retail",
    range(6000, 6800): "Finance / Insurance / Real Estate",
    range(7000, 9000): "Services",
    range(9100, 9730): "Public Administration",
}

### Apply the major-division mapping

This cell defines the mapping function and applies it to the company universe when SIC enrichment is enabled. The final result is saved as the processed ticker / SIC / industry checkpoint used for EDGAR fact extraction.

In [ ]:
def map_sic_major_division(sic):
    if pd.isna(sic):
        return "Unknown"

    sic = int(sic)

    for sic_range, industry in sic_major_division_map.items():
        if sic in sic_range:
            return industry

    return "Other"


if RUN_SIC_ENRICHMENT:
    fundamental_df = fundamental_df.copy()
    fundamental_df["industry"] = fundamental_df["sic"].apply(map_sic_major_division)
    fundamental_df.to_csv(TICKER_SIC_INDUSTRY_PATH, index=False)
    display(fundamental_df["industry"].value_counts())
else:
    print(
        "Major-division mapping write skipped. Set RUN_SIC_ENRICHMENT=True to execute."
    )

## 12. Load the final universe used for EDGAR fact extraction

This cell loads the final processed company universe checkpoint.

At this stage, each company should have its ticker, CIK, SIC code, industry title, and broader industry bucket. This is the company list used by the raw EDGAR fact extraction step.

The purpose of reloading the file is to make the notebook state explicit. The heavy fact download does not depend on a temporary in-memory dataframe from previous cells; it can start from the saved final universe checkpoint.

In [ ]:
fundamental_df = pd.read_csv(TICKER_SIC_INDUSTRY_PATH)
fundamental_df.head()

## 13. Download raw EDGAR financial facts

This is the high-volume data-acquisition step of the notebook.

When enabled, the cell loops through the selected company universe, retrieves financial facts for each ticker, and appends the extracted data to a raw CSV file. The resulting file is the raw accounting fact base used later for model preparation.

This step is guarded carefully because it can become large and slow. A test run can be performed with a small number of tickers using `MAX_TICKERS_FOR_DOWNLOAD`. A full run requires explicit confirmation. This protects the user from accidentally launching a large EDGAR download while reviewing the notebook.

The cell is also resumable. If the raw output already exists, the notebook can detect previously completed tickers and continue with the remaining ones. This matters because a full EDGAR extraction can take a long time and may need to be stopped or resumed.

The output of this step is not yet a model-ready dataset. It is a broad raw fact table: company, filing context, accounting concept, value, fiscal period, and related metadata. Later notebooks and modules decide which concepts become model features.

In [ ]:
if RUN_RAW_EDGAR_FACT_DOWNLOAD:
    if MAX_TICKERS_FOR_DOWNLOAD is None and not CONFIRM_FULL_EDGAR_DOWNLOAD:
        raise RuntimeError(
            "Full EDGAR download is disabled by default. "
            "Set CONFIRM_FULL_EDGAR_DOWNLOAD=True if you intentionally want to run all tickers."
        )

    use_local_storage(True)

    output_file = RAW_FACTS_CSV_PATH

    if not TICKER_SIC_INDUSTRY_PATH.exists():
        raise FileNotFoundError(
            f"Processed ticker universe not found: {TICKER_SIC_INDUSTRY_PATH}. "
            "Run SIC enrichment and industry mapping first."
        )

    fundamental_df = pd.read_csv(TICKER_SIC_INDUSTRY_PATH)

    tickers = fundamental_df["ticker"].dropna().astype(str).unique().tolist()

    if MAX_TICKERS_FOR_DOWNLOAD is not None:
        tickers = tickers[:MAX_TICKERS_FOR_DOWNLOAD]
        print(f"Test run enabled. Downloading raw facts for only {len(tickers)} tickers.")
    else:
        print(f"Full run enabled. Downloading raw facts for {len(tickers)} tickers.")

    # Resume logic.
    if output_file.exists():
        existing = pd.read_csv(output_file)
        completed = set(existing["ticker"].unique())
        print(f"Resuming. Completed tickers: {len(completed)}")
    else:
        existing = pd.DataFrame()
        completed = set()

    rows = []

    for ticker in tqdm(tickers):
        if ticker in completed:
            continue

        try:
            c = Company(ticker)
            facts = c.get_facts()
            df = facts.to_dataframe()

            df["ticker"] = ticker
            df["cik"] = c.cik
            df["company_name"] = c.name
            df["sic"] = c.sic

            # Keep only accounting facts with numeric values.
            df = df[df["numeric_value"].notna()].copy()

            # Optional: keep only annual + quarterly periods.
            df = df[df["fiscal_period"].isin(["FY", "Q1", "Q2", "Q3", "Q4"])]

            rows.append(df)

            # Checkpoint every 25 companies.
            if len(rows) >= 25:
                batch = pd.concat(rows, ignore_index=True)

                if output_file.exists():
                    batch.to_csv(output_file, mode="a", header=False, index=False)
                else:
                    batch.to_csv(output_file, index=False)

                rows = []

        except Exception as e:
            print(f"Error for {ticker}: {e}")
            continue

    # Final save.
    if rows:
        batch = pd.concat(rows, ignore_index=True)

        if output_file.exists():
            batch.to_csv(output_file, mode="a", header=False, index=False)
        else:
            batch.to_csv(output_file, index=False)

    print("Done.")
else:
    print(
        "Raw EDGAR financial fact download skipped. Set RUN_RAW_EDGAR_FACT_DOWNLOAD=True to execute."
    )

## 14. Convert raw CSV to parquet chunks

The raw EDGAR financial facts file can become very large. Reading a large CSV repeatedly is inefficient, especially during modelling and concept-audit work.

This step converts the raw CSV into parquet chunks. Parquet is columnar, compressed, and much faster to query with DuckDB. Chunking also makes the conversion safer because the notebook does not need to load the entire raw CSV into memory at once.

The output folder is `data/raw_financial_facts_parquet/`. Each chunk is saved as a separate parquet file. Later, DuckDB can query the whole folder through a wildcard pattern as if it were one table.

In [ ]:
if RUN_CSV_TO_PARQUET:
    csv_file = RAW_FACTS_CSV_PATH
    output_folder = PARQUET_DIR

    if not csv_file.exists():
        raise FileNotFoundError(
            f"Raw facts CSV not found: {csv_file}. "
            "Run raw EDGAR facts download first."
        )

    output_folder.mkdir(parents=True, exist_ok=True)

    # In test mode, clear old test parquet parts to avoid mixing previous runs.
    if TEST_RUN_MODE:
        for old_file in output_folder.glob("*.parquet"):
            old_file.unlink()
        print(f"TEST_RUN_MODE: cleared old parquet files in {output_folder}")

    chunksize = 250_000

    print("Starting CSV → Parquet chunk conversion...")

    for i, chunk in enumerate(pd.read_csv(csv_file, chunksize=chunksize)):
        output_path = output_folder / f"part_{i:05d}.parquet"
        chunk.to_parquet(output_path, index=False)
        print(f"Saved chunk {i} → {output_path} ({len(chunk):,} rows)")

    print("Done. All chunks converted.")
else:
    print("CSV to parquet conversion skipped. Set RUN_CSV_TO_PARQUET=True to execute.")

## 15. Inspect parquet footprint

This cell checks how many parquet chunks exist and how much disk space they use.

This is a practical operational check. It confirms whether the CSV-to-parquet conversion produced files and gives a rough sense of dataset size. If there are no parquet files, the later DuckDB view cannot be created.

This cell does not change the data. It only verifies the local parquet layer before the notebook moves into DuckDB querying.

In [ ]:
files = glob.glob(str(PARQUET_DIR / "*.parquet"))

print("Parquet files:", len(files))

total_size_gb = sum(os.path.getsize(f) for f in files) / 1e9
print("Total parquet size:", round(total_size_gb, 2), "GB")

## 16. Perform a sanity check

I will check if all files have been generated correctly

In [ ]:
print("Notebook 04 test-run verification")

print("TEST_RUN_MODE:", TEST_RUN_MODE)
print("Fundamental universe exists:", FUNDAMENTAL_UNIVERSE_PATH.exists(), FUNDAMENTAL_UNIVERSE_PATH)
print("Ticker/SIC file exists:", TICKER_CIK_SIC_PATH.exists(), TICKER_CIK_SIC_PATH)
print("Industry file exists:", TICKER_SIC_INDUSTRY_PATH.exists(), TICKER_SIC_INDUSTRY_PATH)
print("Raw facts CSV exists:", RAW_FACTS_CSV_PATH.exists(), RAW_FACTS_CSV_PATH)
print("Parquet dir exists:", PARQUET_DIR.exists(), PARQUET_DIR)
print("Parquet files:", len(list(PARQUET_DIR.glob("*.parquet"))))

if FUNDAMENTAL_UNIVERSE_PATH.exists():
    universe_check = pd.read_csv(FUNDAMENTAL_UNIVERSE_PATH)
    print("Fundamental universe rows:", len(universe_check))

if RAW_FACTS_CSV_PATH.exists():
    raw_check = pd.read_csv(RAW_FACTS_CSV_PATH, nrows=5)
    print("Raw facts sample columns:")
    print(raw_check.columns.tolist())
    display(raw_check.head())

if list(PARQUET_DIR.glob("*.parquet")):
    parquet_sample = pd.read_parquet(sorted(PARQUET_DIR.glob("*.parquet"))[0])
    print("Parquet sample rows:", len(parquet_sample))
    display(parquet_sample.head())

## 17. Create a DuckDB view over parquet files

This step creates a DuckDB view over the parquet folder.

DuckDB allows the notebook to query a large parquet dataset without loading the full dataset into pandas memory. This is a major practical advantage because the EDGAR raw fact dataset can contain millions of rows.

The cell creates or replaces a view called `raw_facts` and then produces a summary of the parquet-backed dataset: total rows, number of tickers, number of accounting concepts, and fiscal-year range. This confirms that the raw EDGAR fact layer is queryable and ready for downstream filtering and concept analysis.

In [ ]:
con = duckdb.connect(str(DUCKDB_PATH))

con.execute(f"""
CREATE OR REPLACE VIEW raw_facts AS
SELECT *
FROM read_parquet('{str(PARQUET_DIR / "*.parquet")}')
""")

summary = con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts,
    MIN(fiscal_year) AS min_year,
    MAX(fiscal_year) AS max_year
FROM raw_facts
""").df()

summary

### Fiscal-year coverage check

This cell shows how many raw fact rows exist by fiscal year. It helps verify whether the dataset covers the intended training period and whether recent years are populated enough for model development.

In [ ]:
con.execute("""
SELECT fiscal_year, COUNT(*) AS n
FROM raw_facts
GROUP BY fiscal_year
ORDER BY fiscal_year
LIMIT 30
""").df()

## 18. Build the model base table

This step creates a filtered DuckDB table for the fiscal years used in model development.

The raw EDGAR fact layer may contain many years. The credit-clustering project uses a defined fiscal-year window so that the training data remains reasonably fresh and comparable. The current notebook filters the raw facts between `TARGET_MIN_FISCAL_YEAR` and `TARGET_MAX_FISCAL_YEAR`.

The resulting table, `credit_model_base`, is still not the final machine-learning table. It is the filtered raw fact base from which concept mapping, feature engineering, and model-ready issuer-year observations can be built.

In [ ]:
con.execute(f"""
CREATE OR REPLACE TABLE credit_model_base AS
SELECT *
FROM raw_facts
WHERE fiscal_year BETWEEN {TARGET_MIN_FISCAL_YEAR} AND {TARGET_MAX_FISCAL_YEAR}
""")

### Model-base summary

This cell checks the size of the filtered fiscal-year table. It reports total rows, distinct tickers, and distinct accounting concepts after applying the target fiscal-year window.

In [ ]:
con.execute("""
SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT ticker) AS tickers,
    COUNT(DISTINCT concept) AS concepts
FROM credit_model_base
""").df()

## 19. Concept coverage audit

This section investigates which XBRL accounting concepts appear most frequently in the filtered EDGAR fact base.

This is necessary because SEC filers do not always use exactly the same XBRL concepts for economically similar items. For example, revenue, equity, capex, and operating income may appear under several different tags depending on the company and filing.

The concept coverage audit helps identify which concepts are broadly available across tickers and therefore good candidates for deterministic concept mapping. This step supports the later feature-engineering process, where raw accounting facts are transformed into standardized financial statement fields.

In [ ]:
concept_counts = con.execute("""
SELECT
    concept,
    label,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ticker) AS ticker_coverage
FROM credit_model_base
GROUP BY concept, label
ORDER BY ticker_coverage DESC
""").df()

### Full concept coverage table

This cell displays the full concept coverage audit. It is useful when selecting or debugging concept mappings because it shows which XBRL concepts are common and which are sparse.

In [ ]:
concept_counts

### Top concept candidates

This cell limits the coverage audit to the top concepts by ticker coverage. It is a practical view for identifying the accounting tags most likely to support robust model features.

In [ ]:
top_concepts = con.execute("""
SELECT
    concept,
    label,
    COUNT(*) AS row_count,
    COUNT(DISTINCT ticker) AS ticker_coverage
FROM credit_model_base
GROUP BY concept, label
ORDER BY ticker_coverage DESC
LIMIT 200
""").df()

top_concepts

### Target concept inspection

This cell defines the key accounting concepts that are especially relevant for credit-risk modelling: assets, liabilities, equity, cash, receivables, inventory, debt, revenue, operating income, net income, interest expense, cash flow, capex, and depreciation/amortization. The coverage check helps confirm which of these concepts are available in the current EDGAR extract.

In [ ]:
target_concepts = [
    "us-gaap:Assets",
    "us-gaap:AssetsCurrent",
    "us-gaap:Liabilities",
    "us-gaap:LiabilitiesCurrent",
    "us-gaap:CashAndCashEquivalentsAtCarryingValue",
    "us-gaap:AccountsReceivableNetCurrent",
    "us-gaap:InventoryNet",
    "us-gaap:PropertyPlantAndEquipmentNet",
    "us-gaap:Goodwill",
    "us-gaap:LongTermDebt",
    "us-gaap:LongTermDebtCurrent",
    "us-gaap:StockholdersEquity",
    "us-gaap:RetainedEarningsAccumulatedDeficit",
    "us-gaap:Revenues",
    "us-gaap:GrossProfit",
    "us-gaap:OperatingIncomeLoss",
    "us-gaap:NetIncomeLoss",
    "us-gaap:InterestExpense",
    "us-gaap:SellingGeneralAndAdministrativeExpense",
    "us-gaap:ResearchAndDevelopmentExpense",
    "us-gaap:NetCashProvidedByUsedInOperatingActivities",
    "us-gaap:PaymentsToAcquirePropertyPlantAndEquipment",
]

target_concepts

## 20. Incremental refresh strategy: new tickers and new fiscal years

This section documents how the EDGAR dataset can be refreshed over time.

The project should not require a full rebuild every time new data becomes available. Instead, the notebook includes logic for detecting new tickers, identifying removed or changed tickers, building a local coverage manifest, and finding companies whose local fiscal-year coverage is stale.

There are two refresh questions:

1. Are there new companies in EDGAR that are not present in the local universe checkpoint?
2. Are there existing companies whose local data does not yet reach the latest target fiscal year?

The refresh strategy is important for future development because the model can be improved or updated only if the underlying data pipeline can be maintained without repeating unnecessary work.

In [ ]:
MANIFEST_PATH = PROCESSED_DIR / "edgar_download_manifest.csv"
SNAPSHOT_DATE = date.today().isoformat()
LATEST_TICKER_SNAPSHOT_PATH = RAW_DIR / f"company_tickers_{SNAPSHOT_DATE}.csv"

if RUN_INCREMENTAL_REFRESH_AUDIT:
    use_local_storage(True)

    latest_tickers = get_company_tickers()
    latest_tickers = (
        latest_tickers.to_pandas()
        if hasattr(latest_tickers, "to_pandas")
        else latest_tickers
    )
    latest_tickers.to_csv(LATEST_TICKER_SNAPSHOT_PATH, index=False)

    current_universe = pd.read_csv(TICKER_SIC_INDUSTRY_PATH)

    existing_tickers = set(current_universe["ticker"].dropna().astype(str))
    latest_ticker_set = set(latest_tickers["ticker"].dropna().astype(str))

    new_tickers = sorted(latest_ticker_set - existing_tickers)
    removed_or_changed_tickers = sorted(existing_tickers - latest_ticker_set)

    print("New tickers:", len(new_tickers))
    print("Removed / changed tickers:", len(removed_or_changed_tickers))
else:
    print(
        "Incremental ticker refresh audit skipped. Set RUN_INCREMENTAL_REFRESH_AUDIT=True to execute."
    )

### Build the local coverage manifest

This cell summarizes the local parquet-backed raw fact dataset by ticker. It records the minimum and maximum fiscal year available for each company and saves the result as a manifest. This does not call EDGAR; it only audits what has already been stored locally.

In [ ]:
# Build or refresh a local coverage manifest from the parquet-backed DuckDB view.
# This does not call EDGAR. It summarizes what is already stored locally.

manifest_query = """
SELECT
    ticker,
    cik,
    company_name,
    sic,
    MIN(fiscal_year) AS min_fiscal_year,
    MAX(fiscal_year) AS max_fiscal_year,
    COUNT(*) AS row_count
FROM raw_facts
GROUP BY ticker, cik, company_name, sic
ORDER BY ticker
"""

if "con" in globals():
    manifest_df = con.execute(manifest_query).df()
    manifest_df["last_audited_at"] = date.today().isoformat()
    manifest_df.to_csv(MANIFEST_PATH, index=False)
    display(manifest_df.head())
else:
    print("DuckDB connection not found. Run the DuckDB view section first.")

### Identify stale tickers

This cell uses the local manifest to identify companies whose stored data does not reach the current target maximum fiscal year. These tickers are candidates for incremental refresh instead of a full rebuild.

In [ ]:
# Identify stale tickers where local data does not yet reach the target maximum fiscal year.

if MANIFEST_PATH.exists():
    manifest = pd.read_csv(MANIFEST_PATH)

    stale_tickers = (
        manifest.loc[
            manifest["max_fiscal_year"] < TARGET_MAX_FISCAL_YEAR,
            "ticker",
        ]
        .dropna()
        .astype(str)
        .tolist()
    )

    print("Stale tickers needing refresh:", len(stale_tickers))
else:
    stale_tickers = []
    print(
        "No manifest found yet. Run the manifest cell after creating the DuckDB raw_facts view."
    )

## 21. Operational notes and final appendix conclusion

This notebook is the data-engineering appendix of the Corporate Credit Clustering Tool. It explains how the EDGAR universe is built, how SIC information is added, how raw financial facts are downloaded, and how the large raw outputs are converted into parquet and DuckDB layers for analysis.

Suggested Git and operational hygiene:

- Keep the notebook under `notebooks/`.
- Keep generated EDGAR files under `data/`.
- Commit small documentation files such as `data/README.md`, but do not commit large raw CSV, parquet, or DuckDB files.
- Keep heavy EDGAR download cells protected by execution flags.
- Use small ticker limits for testing before attempting a full refresh.
- Refresh the universe periodically to detect new tickers.
- Refresh recent fiscal years conservatively, because companies may amend or restate filings.
- Keep SEC access responsible and avoid unnecessary repeated calls when local checkpoints already exist.

The main modelling and scoring workflow is completed in the earlier notebooks. This notebook preserves the raw-data machinery that makes future model refreshes possible. In that sense, Notebook 04 does not change the final model result; it explains and documents the data foundation behind it.